In [1]:
import torch                                 
import torchvision                           
import torchvision.transforms as transforms  
import torchvision.datasets as datasets     
import matplotlib.pyplot as plt              
import numpy as np                           
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim                   
import random
from torch.utils.data import TensorDataset, DataLoader
from dataloader import load_digit_data, load_face_data, get_subset

device = ("cuda" if torch.cuda.is_available() else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")


In [2]:
X_train, y_train = load_digit_data("data/digitdata/trainingimages", "data/digitdata/traininglabels")


In [3]:
print(X_train.shape)
print(y_train.shape)


def split_training_data(percent, x, y):

    train_size = int((percent) * y_train.shape[0])
    indices = torch.randperm(x.shape[0])


    x_train_cut = x[indices[:train_size]]
    y_train_cut = y[indices[:train_size]]

    return torch.from_numpy(x_train_cut), torch.from_numpy(y_train_cut)



(5000, 784)
(5000,)


# Neural Network from Scratch:

The code below initializes weight matrices and bias vectors to create the manual neural network forward pass. The generator is set at the top of the cell for reproducibility. Note at the bottom I toggle all parameter gradients to True, for future automated training. For now, we will implemement the forward and backward propagation from scratch.

There are some things that will be done differently for classification. In class, we were told to perform BCE loss with every single unnormalized logit value and sum the losses across each example. However, I am applying categorical crossentropy loss, with the softmax function:

$$
\text{softmax}(z_i)=\frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

This function will take in a logit vector and turn it into a probability distribution by normalizing the values by the expression above. The result is that the only logit that contributes to the loss of the trianing example is the ground true logit associated with the training example.

In [4]:
'''
Following structure of the neural network:
- Input layer will be the flattened training vectors
- First hidden layer will be of size 784 x 1000
- Second hidden layer will be of size 1000 x 100
- final layer will be of size 100 x 10

I am creating the weight matrices below:
'''


seed = 42
generator = torch.Generator().manual_seed(seed)

hidden_layer_1 = (torch.randn(784, 1000,    generator = generator))
b_1 = (torch.randn(1,1000,                  generator = generator))

hidden_layer_2 = (torch.randn(1000, 100,    generator = generator))
b_2  = (torch.randn(1,100,       generator = generator))

output_layer = (torch.randn(100, 10,    generator = generator))
b_3 = (torch.randn(1,10,       generator = generator))



parameters = [hidden_layer_1, hidden_layer_2, output_layer, b_1, b_2, b_3]
param_list = [(hidden_layer_1, b_1), (hidden_layer_2, b_2), (output_layer, b_3)]

for p in parameters:
    p.requires_grad = True

In [5]:
xc, yc = split_training_data(0.1, X_train, y_train)
print(f"Size of input data before first linear transformation: {xc.shape}")

example_output = xc @ hidden_layer_1

print(f"Size of input data after first linear transformation: {example_output.shape}")

Size of input data before first linear transformation: torch.Size([500, 784])
Size of input data after first linear transformation: torch.Size([500, 1000])


In [14]:
'''
    Consider the forward pass function below. It traverses the tupled list of weight matrices and bias vectors per layer
    and performs the following operation: (f @ W) + b. This operation takes input batch of the previous layer and performs a
    linear transformation on the weight matrix, and then adds the bias vector (one bias value per neuron or row in the matrix).

    Note the activation function being ReLU, which is a squash at zero function. It is a piecewise function where y = 0 if x < 0
    and y = x otherwise. Because of this, the candidate derivative matrix for the ReLU function is just a matrix mask of 0's and 1's
    with a 1 in the derivative position of the value that was non-negative, and a 0 otherwise. Intuitively, ReLU fails to train neurons
    that have negative outputs, as the derivative in the computation graph is zero.

    The following function below also stores the post-activation derivatives of each hidden layer.
'''

def forward_with_computation_graph(p_list, x_batch):
    relu_derivatives = []

    f = x_batch
    for weights, bias in p_list[:-1]:
        pre_activation = (f @ weights) + bias
        f = F.relu(pre_activation)

        relu_derivative = (f > 0)
        relu_derivatives.append(relu_derivative)
    output_weight, output_bias = p_list[-1]

    logits = (f @ output_weight) + output_bias
    return logits, relu_derivatives


In [15]:
out, _ = forward_with_computation_graph(param_list, xc)

print(out.shape)

torch.Size([500, 10])


In [ ]:
#need a dataloader and training function here bra